# LiteLLM Demo

LiteLLM provides a unified interface to 100+ LLM providers with single API.

**Key Features:**
- Provider switching without code changes
- Built-in caching (semantic and exact)
- Fallback chains
- Cost tracking

**Prerequisites:**
```bash
pip install litellm
ollama pull qwen3:4b
```

In [17]:
import litellm
import time

# For Ollama, use the ollama/ prefix
MODEL = "ollama/qwen3:4b"

# Test basic call
try:
    response = litellm.completion(
        model=MODEL,
        messages=[{"role": "user", "content": "Say hello in one word"}],
        api_base="http://localhost:11434"
    )
    print(f"✓ LiteLLM connected: {response.choices[0].message.content[:50]}...")
except Exception as e:
    print(f"✗ Connection failed: {e}")

✓ LiteLLM connected: hi...


---

## 1. Provider Switching

Same code works with any provider - just change the model string.

In [18]:
def call_llm(prompt: str, model: str, **kwargs) -> str:
    """Unified LLM call - works with any provider."""
    response = litellm.completion(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        **kwargs
    )
    return response.choices[0].message.content

# Provider-agnostic model selection
MODELS = {
    "local": "ollama/qwen3:4b",
    # Uncomment with API keys:
    # "openai": "gpt-4o-mini",
    # "anthropic": "claude-3-haiku-20240307",
    # "groq": "groq/llama3-8b-8192",
}

print("Provider-Agnostic LLM Calls")
print("=" * 60)
print("""
# Same function works with any provider:
call_llm("Hello", model="ollama/qwen3:4b")      # Local Ollama
call_llm("Hello", model="gpt-4o-mini")          # OpenAI
call_llm("Hello", model="claude-3-haiku")       # Anthropic
call_llm("Hello", model="groq/llama3-8b-8192")  # Groq

# No code changes needed - just set API keys as env vars
""")

Provider-Agnostic LLM Calls

# Same function works with any provider:
call_llm("Hello", model="ollama/qwen3:4b")      # Local Ollama
call_llm("Hello", model="gpt-4o-mini")          # OpenAI
call_llm("Hello", model="claude-3-haiku")       # Anthropic
call_llm("Hello", model="groq/llama3-8b-8192")  # Groq

# No code changes needed - just set API keys as env vars



---

## 2. Caching for Cost Reduction

In [19]:
from litellm import Cache

# Enable in-memory cache (for demo)
litellm.cache = Cache(type="local")

def timed_call(prompt: str) -> tuple[str, float]:
    """Call LLM and measure time."""
    start = time.time()
    response = litellm.completion(
        model="ollama/qwen3:4b",
        messages=[{"role": "user", "content": prompt}],
        api_base="http://localhost:11434"
    )
    elapsed = time.time() - start
    return response.choices[0].message.content[:100], elapsed

try:
    prompt = "What is 2+2? Answer with just the number."
    
    # First call - no cache
    result1, time1 = timed_call(prompt)
    print(f"First call:  {time1:.2f}s - {result1[:30]}...")
    
    # Second call - should be cached
    result2, time2 = timed_call(prompt)
    print(f"Cached call: {time2:.4f}s - {result2[:30]}...")
    
    print(f"\nSpeedup: {time1/max(time2, 0.001):.0f}x faster with cache")
except Exception as e:
    print(f"Caching demo failed: {e}")

First call:  6.76s - 4...
Cached call: 0.0282s - 4...

Speedup: 240x faster with cache


---

## 3. Fallback Chains

In [20]:
from litellm import Router

# Configure router with fallback
router_config = {
    "model_list": [
        {
            "model_name": "main",
            "litellm_params": {
                "model": "ollama/qwen3:4b",
                "api_base": "http://localhost:11434"
            }
        },
        # Fallback models (if you have API keys)
        # {
        #     "model_name": "main",
        #     "litellm_params": {
        #         "model": "gpt-4o-mini"
        #     }
        # },
    ],
    "routing_strategy": "simple-shuffle",
    "num_retries": 2,
    "timeout": 30
}

print("Fallback Configuration Pattern")
print("=" * 60)
print("""
# Router automatically fails over to next provider
router = Router(model_list=[
    {"model_name": "main", "litellm_params": {"model": "gpt-4o"}},
    {"model_name": "main", "litellm_params": {"model": "claude-3-opus"}},  # Fallback
])

# Calls first available, falls back on error
response = router.completion(model="main", messages=[...])
""")

Fallback Configuration Pattern

# Router automatically fails over to next provider
router = Router(model_list=[
    {"model_name": "main", "litellm_params": {"model": "gpt-4o"}},
    {"model_name": "main", "litellm_params": {"model": "claude-3-opus"}},  # Fallback
])

# Calls first available, falls back on error
response = router.completion(model="main", messages=[...])



---

## 4. Cost Tracking

In [21]:
from litellm import completion_cost

# Make a call and track cost
try:
    response = litellm.completion(
        model="ollama/qwen3:4b",
        messages=[{"role": "user", "content": "Write a haiku about coding"}],
        api_base="http://localhost:11434"
    )
    
    # Note: Local models don't have costs, but API models do
    print("Cost Tracking (for API models)")
    print("=" * 60)
    print(f"Response: {response.choices[0].message.content[:100]}...")
    print(f"Usage: {response.usage}")
    
    # For API models, you'd see actual costs:
    # cost = completion_cost(response)
    # print(f"Cost: ${cost:.6f}")
    
    print("""
# For API models:
response = litellm.completion(model="gpt-4o-mini", ...)
cost = completion_cost(response)
print(f"This call cost: ${cost:.6f}")
""")
except Exception as e:
    print(f"Cost tracking demo: {e}")

Cost Tracking (for API models)
Response: Bugs hide in the code  
Debugging finds the path through  
Logic wins the game...
Usage: Usage(completion_tokens=978, prompt_tokens=20, total_tokens=998, completion_tokens_details=None, prompt_tokens_details=None)

# For API models:
response = litellm.completion(model="gpt-4o-mini", ...)
cost = completion_cost(response)
print(f"This call cost: ${cost:.6f}")



---

## 5. LiteLLM + Haystack Integration

Use LiteLLM as the generator component in Haystack pipelines for provider flexibility.

In [22]:
from haystack import component
import litellm

@component
class LiteLLMGenerator:
    """
    Haystack generator component using LiteLLM for provider flexibility.
    
    Benefits:
    - Swap providers without changing pipeline code
    - Automatic fallbacks on provider failure
    - Cost tracking across providers
    """
    
    def __init__(self, model: str = "ollama/qwen3:4b", fallbacks: list = None, **kwargs):
        self.model = model
        self.fallbacks = fallbacks or []
        self.kwargs = kwargs
    
    @component.output_types(replies=list[str])
    def run(self, prompt: str):
        response = litellm.completion(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            fallbacks=self.fallbacks,
            **self.kwargs
        )
        return {"replies": [response.choices[0].message.content]}

# Test the component
try:
    generator = LiteLLMGenerator(
        model="ollama/qwen3:4b",
        api_base="http://localhost:11434"
    )
    
    result = generator.run(prompt="What is 2+2? Answer briefly.")
    print("LiteLLM + Haystack Integration")
    print("=" * 60)
    print(f"Response: {result['replies'][0][:100]}...")
    print("""
# Use in Haystack pipeline:
from haystack import Pipeline

pipeline = Pipeline()
pipeline.add_component("llm", LiteLLMGenerator(
    model="claude-3-5-sonnet-20241022",
    fallbacks=["gpt-4o", "ollama/qwen3:4b"]
))
""")
except Exception as e:
    print(f"Haystack integration demo: {e}")

LiteLLM + Haystack Integration
Response: 4...

# Use in Haystack pipeline:
from haystack import Pipeline

pipeline = Pipeline()
pipeline.add_component("llm", LiteLLMGenerator(
    model="claude-3-5-sonnet-20241022",
    fallbacks=["gpt-4o", "ollama/qwen3:4b"]
))



---

## Summary

**LiteLLM Value Propositions:**

| Feature | Benefit |
|---------|---------|
| Unified API | Same code for 100+ providers |
| Caching | 20-60% cost reduction |
| Fallbacks | Zero-downtime provider switching |
| Cost Tracking | Budget monitoring per call |

**Production Setup:**
```bash
# Run as proxy server
litellm --model ollama/qwen3:4b --port 8000

# All apps call localhost:8000 with OpenAI-compatible API
```

**When to use LiteLLM:**
- Multi-provider flexibility needed
- Cost optimization important
- Standardized interface for team